In [2]:
import os
import sys
from pathlib import Path
current_dir = Path.cwd() 
project_root = os.path.dirname(os.path.dirname(current_dir))     # 项目根目录
sys.path.insert(0, project_root)
data_output_dir = os.path.join(project_root, "data")
test_data_path = os.path.join(data_output_dir, "test_data.json")
data_path = os.path.join(data_output_dir, "output/test_results_vllm.json")
print(data_path,test_data_path)

/mnt/workspace/Graduate/ToxiCN_lora_rag/data/output/test_results_vllm.json /mnt/workspace/Graduate/ToxiCN_lora_rag/data/test_data.json


In [3]:
import json
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

data = load_json(data_path)
print(f"pred数据长度：{len(data)}")
test_data = load_json(test_data_path)
print(f"gold数据长度：{len(test_data)}")


pred数据长度：1605
gold数据长度：1605


In [16]:
from src.evaluate import evaluate
from src.data import str_to_quadruples
pred_data = []
gold_data = [{"id": d["id"], "content": d["content"], "quadruples": d["quadruples"]} for d in test_data]
for d in data:
    pred = str_to_quadruples(d['output'])
    pred_data.append({
        'id': d['id'],
        'content': d['content'],
        'output': d['output'],
        'quadruples': pred,
    })
score = evaluate(gold_data, pred_data)


In [17]:

print(score)
print(f"Hard Score: {score['hard_f1']:.4f}")
print(f"Soft Score: {score['soft_f1']:.4f}")
print(f"Score: {score['avg_f1']:.4f}")

{'hard_precision': 0.1616862326574173, 'hard_recall': 0.15930599369085174, 'hard_f1': 0.1604872881355932, 'soft_precision': 0.30362860192102453, 'soft_recall': 0.29915878023133546, 'soft_f1': 0.30137711864406785, 'avg_f1': 0.23093220338983053}
Hard Score: 0.1605
Soft Score: 0.3014
Score: 0.2309


In [10]:
# Quadruple Hard Matching

import json

# Calculate precision, recall, and F1 score
def calculate_metrics(predictions):
    """Calculate precision, recall, and F1 score for model predictions"""
    
    true_positives = 0  # True positives: Number of correctly predicted quadruples
    predicted_positives = 0  # Predicted positives: Total number of quadruples predicted by the model
    actual_positives = 0  # Actual positives: Total number of quadruples in the ground truth

    for item in predictions:
        # Split target and predicted quadruples
        target_quadruples = item["ground_truth"].split("[SEP]")
        prediction_quadruples = item["output"].split("[SEP]")

        target_quadruples = [t.replace('[END]', '').strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.replace('[END]', '').strip() for p in prediction_quadruples if p.strip()]
        
        # Remove trailing whitespace from each quadruple to avoid comparison issues
        target_quadruples = [t.strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.strip() for p in prediction_quadruples if p.strip()]
        
        actual_positives += len(target_quadruples)  # Update total actual positives
        predicted_positives += len(prediction_quadruples)  # Update total predicted positives

        # Compare each predicted quadruple with target quadruples
        for pred in prediction_quadruples:
            for t in target_quadruples:
                if pred == t:  # If predicted quadruple exactly matches target quadruple
                    true_positives += 1
                    target_quadruples.remove(t)  # Ensure each target quadruple is matched only once
                    break  # Exit inner loop once a match is found
        
    # Calculate precision
    precision = true_positives / predicted_positives if predicted_positives > 0 else 0
    
    # Calculate recall
    recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    # Calculate F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score

# Load data
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Main function
def main(input_json_path):
    # 1. Load data
    data = load_data(input_json_path)

    # 2. Calculate metrics
    precision, recall, f1_score = calculate_metrics(data)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

if __name__ == "__main__":
    main(data_path)

Precision: 0.1602
Recall: 0.1593
F1 Score: 0.1598


In [18]:
# Quadruple Soft Matching

import json
from difflib import SequenceMatcher

# Calculate precision, recall, and F1 score (soft matching)
def calculate_metrics(predictions):
    """Calculate precision, recall, and F1 score for model predictions (soft matching)"""
    
    true_positives = 0  # True positives: Number of correctly predicted quadruples
    predicted_positives = 0  # Predicted positives: Total number of quadruples predicted by the model
    actual_positives = 0  # Actual positives: Total number of quadruples in the ground truth

    for item in predictions:
        # Split target and predicted quadruples
        target_quadruples = item["ground_truth"].split("[SEP]")
        prediction_quadruples = item["output"].split("[SEP]")

        target_quadruples = [t.replace('[END]', '').strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.replace('[END]', '').strip() for p in prediction_quadruples if p.strip()]
        
        # Remove trailing whitespace from each quadruple to avoid comparison issues
        target_quadruples = [t.strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.strip() for p in prediction_quadruples if p.strip()]
        
        actual_positives += len(target_quadruples)  # Update total actual positives
        predicted_positives += len(prediction_quadruples)  # Update total predicted positives
        
        # Compare each predicted quadruple with target quadruples
        for pred in prediction_quadruples:
            pred_parts = pred.split("|")
            target_parts = None
            
            matched = False  # Flag to indicate if a matching target quadruple is found
            
            # Find matching target quadruple
            for t in target_quadruples:
                target_parts = t.split("|")
                
                if len(target_parts) == 4 and len(pred_parts) == 4:
                    target_entity, target_speech, target_group, target_hate = target_parts
                    pred_entity, pred_speech, pred_group, pred_hate = pred_parts
                    
                    # Use soft matching for the first two components
                    similarity = SequenceMatcher(None, target_entity, pred_entity).ratio()
                    similarity_speech = SequenceMatcher(None, target_speech, pred_speech).ratio()
                    
                    # If similarity for first two components is >= 0.5 and last two components match exactly
                    if similarity >= 0.5 and similarity_speech >= 0.5 and target_group == pred_group and target_hate == pred_hate:
                        true_positives += 1
                        target_quadruples.remove(t)  # Ensure each target quadruple is matched only once
                        matched = True  # Mark as matched
                        break  # Exit inner loop and move to next predicted quadruple
        
    # Calculate precision
    precision = true_positives / predicted_positives if predicted_positives > 0 else 0
    
    # Calculate recall
    recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    # Calculate F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score

# Load data
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Main function
def main(input_json_path):
    # 1. Load data
    data = load_data(input_json_path)

    # 2. Calculate metrics
    precision, recall, f1_score = calculate_metrics(data)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

# Run main function
if __name__ == "__main__":
    main(data_path)

Precision: 0.3141
Recall: 0.3123
F1 Score: 0.3132


In [20]:
# Triple Hard Matching
import json

# Calculate precision, recall, and F1 score
def calculate_metrics(predictions):
    """Calculate precision, recall, and F1 score for model predictions"""
    
    true_positives = 0  # True positives: Number of correctly predicted quadruples
    predicted_positives = 0  # Predicted positives: Total number of quadruples predicted by the model
    actual_positives = 0  # Actual positives: Total number of quadruples in the ground truth

    for item in predictions:
        # Split target and predicted quadruples
        target_quadruples = item["ground_truth"].split("[SEP]")
        prediction_quadruples = item["output"].split("[SEP]")

        target_quadruples = [t.replace('[END]', '').strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.replace('[END]', '').strip() for p in prediction_quadruples if p.strip()]
        
        # Remove trailing whitespace from each quadruple to avoid comparison issues
        target_quadruples = [t.strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.strip() for p in prediction_quadruples if p.strip()]
        
        actual_positives += len(target_quadruples)  # Update total actual positives
        predicted_positives += len(prediction_quadruples)  # Update total predicted positives

        # Compare each predicted quadruple with target quadruples
        for pred in prediction_quadruples:
            pred_parts = pred.split("|")
            
            for t in target_quadruples:
                target_parts = t.split("|")
                
                # Ensure the number of elements in quadruples is correct, then compare the first two and last elements
                if len(target_parts) == 4 and len(pred_parts) == 4:
                    target_entity, target_speech, _, target_hate = target_parts
                    pred_entity, pred_speech, _, pred_hate = pred_parts
                    
                    # If the first two elements and the last element are identical, consider it a match
                    if (target_entity == pred_entity and 
                        target_speech == pred_speech and
                        target_hate == pred_hate):
                        
                        true_positives += 1
                        target_quadruples.remove(t)  # Ensure each target quadruple is matched only once
                        break  # Exit inner loop once a match is found
        
    # Calculate precision
    precision = true_positives / predicted_positives if predicted_positives > 0 else 0
    
    # Calculate recall
    recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    # Calculate F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score

# Load data
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Main function
def main(input_json_path):
    # 1. Load data
    data = load_data(input_json_path)

    # 2. Calculate metrics
    precision, recall, f1_score = calculate_metrics(data)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

if __name__ == "__main__":
    main(data_path)

Precision: 0.1904
Recall: 0.1893
F1 Score: 0.1898


In [21]:
# Triple Soft Matching
import json
from difflib import SequenceMatcher

# Calculate precision, recall, and F1 score (soft matching)
def calculate_metrics(predictions):
    """Calculate precision, recall, and F1 score for model predictions (soft matching)"""
    
    true_positives = 0  # True positives: Number of correctly predicted quadruples
    predicted_positives = 0  # Predicted positives: Total number of quadruples predicted by the model
    actual_positives = 0  # Actual positives: Total number of quadruples in the ground truth

    for item in predictions:
        # Split target and predicted quadruples
        target_quadruples = item["ground_truth"].split("[SEP]")
        prediction_quadruples = item["output"].split("[SEP]")

        target_quadruples = [t.replace('[END]', '').strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.replace('[END]', '').strip() for p in prediction_quadruples if p.strip()]
        
        # Remove trailing whitespace from each quadruple to avoid comparison issues
        target_quadruples = [t.strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.strip() for p in prediction_quadruples if p.strip()]
        
        actual_positives += len(target_quadruples)  # Update total actual positives
        predicted_positives += len(prediction_quadruples)  # Update total predicted positives
        
        # Compare each predicted quadruple with target quadruples
        for pred in prediction_quadruples:
            pred_parts = pred.split("|")
            target_parts = None
            
            matched = False  # Flag to indicate if a matching target quadruple is found
            
            # Find matching target quadruple
            for t in target_quadruples:
                target_parts = t.split("|")
                
                # If the first two elements have similarity > 0.5 and the last element matches exactly, consider it a match
                if len(target_parts) == 4 and len(pred_parts) == 4:
                    target_entity, target_speech, _, target_hate = target_parts  # Ignore the third element
                    pred_entity, pred_speech, _, pred_hate = pred_parts  # Ignore the third element
                    
                    # Use soft matching for the first two elements
                    similarity = SequenceMatcher(None, target_entity, pred_entity).ratio()
                    similarity_speech = SequenceMatcher(None, target_speech, pred_speech).ratio()
                    
                    # If the first two elements have similarity > 0.5 and the last element matches exactly
                    if similarity >= 0.5 and similarity_speech >= 0.5 and target_hate == pred_hate:
                        true_positives += 1
                        target_quadruples.remove(t)  # Ensure each target quadruple is matched only once
                        matched = True  # Mark as matched
                        break  # Exit inner loop and move to next predicted quadruple
        
    # Calculate precision
    precision = true_positives / predicted_positives if predicted_positives > 0 else 0
    
    # Calculate recall
    recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    # Calculate F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score

# Load data
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Main function
def main(input_json_path):
    # 1. Load data
    data = load_data(input_json_path)

    # 2. Calculate metrics
    precision, recall, f1_score = calculate_metrics(data)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

if __name__ == "__main__":
    main(data_path)

Precision: 0.4019
Recall: 0.3996
F1 Score: 0.4007


In [22]:
# Pair Hard Matching
import json

# Calculate precision, recall, and F1 score
def calculate_metrics(predictions):
    """Calculate precision, recall, and F1 score for model predictions"""
    
    true_positives = 0  # True positives: Number of correctly predicted pairs
    predicted_positives = 0  # Predicted positives: Total number of pairs predicted by the model
    actual_positives = 0  # Actual positives: Total number of pairs in the ground truth

    for item in predictions:
        # Split target and predicted quadruples
        target_quadruples = item["ground_truth"].split("[SEP]")
        prediction_quadruples = item["output"].split("[SEP]")

        target_quadruples = [t.replace('[END]', '').strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.replace('[END]', '').strip() for p in prediction_quadruples if p.strip()]
        
        # Remove trailing whitespace from each quadruple to avoid comparison issues
        target_quadruples = [t.strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.strip() for p in prediction_quadruples if p.strip()]
        
        actual_positives += len(target_quadruples)  # Update total actual positives
        predicted_positives += len(prediction_quadruples)  # Update total predicted positives

        # Compare each predicted quadruple with target quadruples
        for pred in prediction_quadruples:
            pred_parts = pred.split("|")
            
            for t in target_quadruples:
                target_parts = t.split("|")
                
                # Ensure the number of elements in quadruples is correct, then compare the first two elements
                if len(target_parts) == 4 and len(pred_parts) == 4:
                    target_entity, target_speech, _, _ = target_parts
                    pred_entity, pred_speech, _, _ = pred_parts
                    
                    # If the first two elements are identical, consider it a match
                    if (target_entity == pred_entity and 
                        target_speech == pred_speech):
                        
                        true_positives += 1
                        target_quadruples.remove(t)  # Ensure each target quadruple is matched only once
                        break  # Exit inner loop once a matching pair is found
        
    # Calculate precision
    precision = true_positives / predicted_positives if predicted_positives > 0 else 0
    
    # Calculate recall
    recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    # Calculate F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score

# Load data
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Main function
def main(input_json_path):
    # 1. Load data
    data = load_data(input_json_path)

    # 2. Calculate metrics
    precision, recall, f1_score = calculate_metrics(data)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

if __name__ == "__main__":
    main(data_path)

Precision: 0.2528
Recall: 0.2513
F1 Score: 0.2520


In [23]:
# Pair Soft Matching
import json
from difflib import SequenceMatcher

# Calculate precision, recall, and F1 score (soft matching)
def calculate_metrics(predictions):
    """Calculate precision, recall, and F1 score for model predictions (soft matching)"""
    
    true_positives = 0  # True positives: Number of correctly predicted doublets
    predicted_positives = 0  # Predicted positives: Total number of doublets predicted by the model
    actual_positives = 0  # Actual positives: Total number of doublets in the ground truth

    for item in predictions:
        # Split target and predicted quadruples
        target_quadruples = item["ground_truth"].split("[SEP]")
        prediction_quadruples = item["output"].split("[SEP]")

        target_quadruples = [t.replace('[END]', '').strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.replace('[END]', '').strip() for p in prediction_quadruples if p.strip()]
        
        # Remove trailing whitespace from each quadruple to avoid comparison issues
        target_quadruples = [t.strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.strip() for p in prediction_quadruples if p.strip()]
        
        actual_positives += len(target_quadruples)  # Update total actual positives
        predicted_positives += len(prediction_quadruples)  # Update total predicted positives
        
        # Compare each predicted quadruple with target quadruples
        for pred in prediction_quadruples:
            pred_parts = pred.split("|")
            target_parts = None
            
            matched = False  # Flag to indicate if a matching target quadruple is found
            
            # Find matching target quadruple
            for t in target_quadruples:
                target_parts = t.split("|")
                
                # If the similarity of the first two elements exceeds 0.5, consider it a match
                if len(target_parts) == 4 and len(pred_parts) == 4:
                    target_entity, target_speech, _, _ = target_parts  # Ignore the last two elements
                    pred_entity, pred_speech, _, _ = pred_parts  # Ignore the last two elements
                    
                    # Use soft matching for the first two elements
                    similarity = SequenceMatcher(None, target_entity, pred_entity).ratio()
                    similarity_speech = SequenceMatcher(None, target_speech, pred_speech).ratio()
                    
                    # If the similarity of the first two elements is greater than 0.5, consider it a match
                    if similarity >= 0.5 and similarity_speech >= 0.5:
                        true_positives += 1
                        target_quadruples.remove(t)  # Ensure each target quadruple is matched only once
                        matched = True  # Mark as matched
                        break  # Exit inner loop and move to next predicted quadruple
        
    # Calculate precision
    precision = true_positives / predicted_positives if predicted_positives > 0 else 0
    
    # Calculate recall
    recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    # Calculate F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score

# Load data
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Main function
def main(input_json_path):
    # 1. Load data
    data = load_data(input_json_path)

    # 2. Calculate metrics
    precision, recall, f1_score = calculate_metrics(data)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

if __name__ == "__main__":
    # Path to your test file
    main(data_path)

Precision: 0.5336
Recall: 0.5305
F1 Score: 0.5320


In [24]:
# Single Element Hard Matching
import json

# Calculate precision, recall, and F1 score (hard matching)
def calculate_metrics(predictions):
    """Calculate precision, recall, and F1 score for model predictions (hard matching)"""
    
    true_positives = 0  # True positives: Number of correctly predicted elements
    predicted_positives = 0  # Predicted positives: Total number of elements predicted by the model
    actual_positives = 0  # Actual positives: Total number of elements in the ground truth

    for item in predictions:
        # Split target and predicted quadruples
        target_quadruples = item["ground_truth"].split("[SEP]")
        prediction_quadruples = item["output"].split("[SEP]")

        target_quadruples = [t.replace('[END]', '').strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.replace('[END]', '').strip() for p in prediction_quadruples if p.strip()]
        
        # Remove trailing whitespace from each quadruple to avoid comparison issues
        target_quadruples = [t.strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.strip() for p in prediction_quadruples if p.strip()]
        
        actual_positives += len(target_quadruples)  # Update total actual positives
        predicted_positives += len(prediction_quadruples)  # Update total predicted positives
        
        # Compare each predicted quadruple with target quadruples
        for pred in prediction_quadruples:
            pred_parts = pred.split("|")
            
            # Find matching target quadruple
            for t in target_quadruples:
                target_parts = t.split("|")
                
                # If the second element of the predicted and target quadruples exactly match, consider it a match
                if len(target_parts) == 4 and len(pred_parts) == 4:
                    _, target_speech, _, _ = target_parts  
                    _, pred_speech, _, _ = pred_parts 

                    # If the second element exactly matches
                    if target_speech == pred_speech:
                        true_positives += 1
                        target_quadruples.remove(t)  # Ensure each target quadruple is matched only once
                        break  # Exit inner loop and move to next predicted quadruple
        
    # Calculate precision
    precision = true_positives / predicted_positives if predicted_positives > 0 else 0
    
    # Calculate recall
    recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    # Calculate F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score

# Load data
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Main function
def main(input_json_path):
    # 1. Load data
    data = load_data(input_json_path)

    # 2. Calculate metrics
    precision, recall, f1_score = calculate_metrics(data)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

if __name__ == "__main__":
    main(data_path)

Precision: 0.2967
Recall: 0.2950
F1 Score: 0.2958


In [25]:
# Single Element Soft Matching
import json
from difflib import SequenceMatcher

# Calculate precision, recall, and F1 score (soft matching)
def calculate_metrics(predictions):
    """Calculate precision, recall, and F1 score for model predictions (soft matching)"""
    
    true_positives = 0  # True positives: Number of correctly predicted elements
    predicted_positives = 0  # Predicted positives: Total number of elements predicted by the model
    actual_positives = 0  # Actual positives: Total number of elements in the ground truth

    for item in predictions:
        # Split target and predicted quadruples
        target_quadruples = item["ground_truth"].split("[SEP]")
        prediction_quadruples = item["output"].split("[SEP]")

        target_quadruples = [t.replace('[END]', '').strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.replace('[END]', '').strip() for p in prediction_quadruples if p.strip()]
        
        # Remove trailing whitespace from each quadruple to avoid comparison issues
        target_quadruples = [t.strip() for t in target_quadruples if t.strip()]
        prediction_quadruples = [p.strip() for p in prediction_quadruples if p.strip()]
        
        actual_positives += len(target_quadruples)  # Update total actual positives
        predicted_positives += len(prediction_quadruples)  # Update total predicted positives
        
        # Compare each predicted quadruple with target quadruples
        for pred in prediction_quadruples:
            pred_parts = pred.split("|")
            target_parts = None
            
            matched = False  # Flag to indicate if a matching target quadruple is found
            
            # Find matching target quadruple
            for t in target_quadruples:
                target_parts = t.split("|")
                
                # If the similarity of the first element exceeds 0.5, consider it a match
                if len(target_parts) == 4 and len(pred_parts) == 4:
                    _, target_speech, _, _ = target_parts  
                    _, pred_speech, _, _ = pred_parts 
                    
                    # Use soft matching for the first element
                    similarity = SequenceMatcher(None, target_speech, pred_speech).ratio()
                    # If the similarity of the first element is greater than 0.5, consider it a match
                    if similarity >= 0.5:
                        true_positives += 1
                        target_quadruples.remove(t)  # Ensure each target quadruple is matched only once
                        matched = True  # Mark as matched
                        break  # Exit inner loop and move to next predicted quadruple
        
    # Calculate precision
    precision = true_positives / predicted_positives if predicted_positives > 0 else 0
    
    # Calculate recall
    recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    # Calculate F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score

# Load data
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Main function
def main(input_json_path):
    # 1. Load data
    data = load_data(input_json_path)

    # 2. Calculate metrics
    precision, recall, f1_score = calculate_metrics(data)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

if __name__ == "__main__":
    main(data_path)

Precision: 0.6351
Recall: 0.6314
F1 Score: 0.6333
